# 3D Gaussian Splatting — Step by Step

3DGS의 내부 동작을 단계별로 실행하며 이해하는 실습 노트북.

1. SfM sparse point 로딩
2. Gaussian 초기화 (속성 확인)
3. 학습 루프 (500 iteration 단위로 스냅샷)
4. Densification에 의한 포인트 수 변화
5. Train/Test 뷰 렌더링 비교

In [ ]:
# === 셀 1: 설정 & import ===
import os, sys, glob, torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from random import randint
from tqdm import tqdm

# ========== 여기만 수정 ==========
SCENE_NAME = "bicycle"
IMAGE_SCALE = 4  # 1=원본, 2=1/2, 4=1/4, 8=1/8
TOTAL_ITERATIONS = 5000  # 전체 학습 iteration (교육용으로 짧게)
SNAPSHOT_INTERVAL = 500  # 스냅샷 간격
# ================================

BASE_DIR = "/home/daeho/storage/3dgs_sba"
GS_REPO = os.path.join(BASE_DIR, "repos", "gaussian-splatting")
DATASET_DIR = os.path.join(BASE_DIR, "datasets", "mipnerf360", SCENE_NAME)

# gaussian-splatting 모듈 import
sys.path.insert(0, GS_REPO)
os.chdir(GS_REPO)

from scene import Scene, GaussianModel
from gaussian_renderer import render
from utils.loss_utils import l1_loss, ssim
from utils.general_utils import safe_state, get_expon_lr_func
from utils.image_utils import psnr
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM = True
except:
    FUSED_SSIM = False

print(f"Scene: {SCENE_NAME}, Scale: 1/{IMAGE_SCALE}, Iterations: {TOTAL_ITERATIONS}")
print(f"Snapshots every {SNAPSHOT_INTERVAL} iterations")
print(f"torch {torch.__version__}, CUDA {torch.version.cuda}, GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# === 셀 2: Scene & GaussianModel 로딩 ===
# ArgumentParser로 설정 구성 (train.py와 동일한 방식)

parser = ArgumentParser()
lp = ModelParams(parser)
op = OptimizationParams(parser)
pp = PipelineParams(parser)

args = parser.parse_args([
    "-s", DATASET_DIR,
    "-m", os.path.join(BASE_DIR, "output", f"{SCENE_NAME}_step_by_step"),
    "--resolution", str(IMAGE_SCALE),
    "--eval",
    "--iterations", str(TOTAL_ITERATIONS),
    "--densify_grad_threshold", "0.001",
])

safe_state(True)  # quiet mode

dataset = lp.extract(args)
opt = op.extract(args)
pipe = pp.extract(args)

# GaussianModel 생성 & Scene 로딩
gaussians = GaussianModel(dataset.sh_degree)
scene = Scene(dataset, gaussians)
gaussians.training_setup(opt)

bg_color = [0, 0, 0]
background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

# Train/Test 카메라
train_cameras = scene.getTrainCameras()
test_cameras = scene.getTestCameras()

print(f"\n=== Scene Loaded ===")
print(f"  Train cameras: {len(train_cameras)}")
print(f"  Test cameras:  {len(test_cameras)}")
print(f"  Initial Gaussians: {gaussians.get_xyz.shape[0]:,}")
print(f"  Scene extent: {scene.cameras_extent:.4f}")

## SfM Sparse Points & Gaussian 초기화

3DGS는 COLMAP의 sparse 3D 포인트를 **초기 Gaussian 위치**로 사용합니다.
각 포인트는 다음 속성을 가진 3D Gaussian으로 초기화됩니다:

| 속성 | 초기값 | 설명 |
|------|--------|------|
| **Position (xyz)** | SfM 포인트 좌표 | Gaussian 중심 위치 |
| **Color (SH)** | SfM 포인트 색상 → SH 변환 | Spherical Harmonics 계수 |
| **Opacity** | 0.1 (inverse sigmoid) | 투명도 (0~1) |
| **Scale** | nearest neighbor 거리 | Gaussian 크기 (3축) |
| **Rotation** | identity [1,0,0,0] | 회전 (quaternion) |

In [ ]:
# === 셀 3: Gaussian 속성 디버그 (처음 5개) ===
# 초기화 직후 Gaussian들의 속성을 확인합니다.

N_SHOW = 5
n_total = gaussians.get_xyz.shape[0]

print(f"=== 초기 Gaussian 속성 (처음 {N_SHOW}개 / 전체 {n_total:,}개) ===\n")

with torch.no_grad():
    xyz = gaussians.get_xyz[:N_SHOW].cpu().numpy()
    opacity = gaussians.get_opacity[:N_SHOW].cpu().numpy()
    scales = gaussians.get_scaling[:N_SHOW].cpu().numpy()
    rotations = gaussians.get_rotation[:N_SHOW].cpu().numpy()
    # SH DC 계수 → RGB 근사 (SH2RGB)
    sh_dc = gaussians.get_features_dc[:N_SHOW, :, 0].cpu().numpy()  # (N, 3)
    colors_approx = np.clip(sh_dc * 0.28209479177 + 0.5, 0, 1)  # C0 * SH_DC + 0.5

for i in range(N_SHOW):
    print(f"--- Gaussian #{i} ---")
    print(f"  Position (xyz)  : [{xyz[i,0]:.4f}, {xyz[i,1]:.4f}, {xyz[i,2]:.4f}]")
    print(f"  Opacity (alpha) : {opacity[i,0]:.4f}")
    print(f"  Scale (3-axis)  : [{scales[i,0]:.6f}, {scales[i,1]:.6f}, {scales[i,2]:.6f}]")
    print(f"  Rotation (quat) : [{rotations[i,0]:.4f}, {rotations[i,1]:.4f}, {rotations[i,2]:.4f}, {rotations[i,3]:.4f}]")
    print(f"  Color (RGB)     : [{colors_approx[i,0]:.3f}, {colors_approx[i,1]:.3f}, {colors_approx[i,2]:.3f}]")
    print()

# 시각화: 속성 분포
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].hist(gaussians.get_opacity.detach().cpu().numpy(), bins=50, color="steelblue", edgecolor="white")
axes[0].set_title(f"Opacity Distribution\n(all {n_total:,} gaussians)")
axes[0].set_xlabel("Opacity")

scale_norms = gaussians.get_scaling.detach().cpu().numpy().mean(axis=1)
axes[1].hist(scale_norms, bins=50, color="coral", edgecolor="white")
axes[1].set_title("Scale (mean of 3-axis)")
axes[1].set_xlabel("Scale")

axes[2].hist(colors_approx.flatten(), bins=50, color="green", edgecolor="white")
axes[2].set_title(f"Color Distribution\n(first {N_SHOW})")
axes[2].set_xlabel("RGB value")

# 3D 포인트 위치 (서브샘플)
xyz_all = gaussians.get_xyz.detach().cpu().numpy()
sub = max(1, len(xyz_all) // 3000)
axes[3].scatter(xyz_all[::sub, 0], xyz_all[::sub, 2], s=0.3, alpha=0.5, c="purple")
axes[3].set_title(f"Initial Point Cloud\n({n_total:,} points)")
axes[3].set_xlabel("X"); axes[3].set_ylabel("Z")
axes[3].set_aspect("equal")

plt.tight_layout()
plt.show()

In [ ]:
# === 셀 4: 초기 렌더링 (학습 전) ===
# 학습 전 Gaussian으로 렌더링하면 어떻게 보이는지 확인

# 고정 시점 선택 (학습 내내 같은 시점으로 비교)
fixed_train_cam = train_cameras[0]
fixed_test_cam = test_cameras[0]

with torch.no_grad():
    train_render = render(fixed_train_cam, gaussians, pipe, background)["render"]
    test_render = render(fixed_test_cam, gaussians, pipe, background)["render"]
    train_gt = fixed_train_cam.original_image.cuda()
    test_gt = fixed_test_cam.original_image.cuda()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].imshow(train_render.permute(1, 2, 0).clamp(0, 1).cpu().numpy())
axes[0, 0].set_title("Train View - Render (iter 0)")
axes[0, 1].imshow(train_gt.permute(1, 2, 0).clamp(0, 1).cpu().numpy())
axes[0, 1].set_title("Train View - GT")
axes[1, 0].imshow(test_render.permute(1, 2, 0).clamp(0, 1).cpu().numpy())
axes[1, 0].set_title("Test View - Render (iter 0)")
axes[1, 1].imshow(test_gt.permute(1, 2, 0).clamp(0, 1).cpu().numpy())
axes[1, 1].set_title("Test View - GT")

for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Before Training (iteration 0)", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Fixed train view: {fixed_train_cam.image_name}")
print(f"Fixed test view:  {fixed_test_cam.image_name}")

## 학습 루프 (스냅샷 포함)

학습 과정에서 500 iteration마다:
- 현재 Gaussian 개수 기록
- 고정 시점(Train/Test)에서 렌더링 스냅샷 저장
- Loss, PSNR 기록

**핵심 과정:**
1. 랜덤 train 카메라 선택 → 렌더링 → GT와 비교 → Loss 계산 → 역전파
2. Densification (500~15000 iter): gradient가 큰 Gaussian을 clone/split
3. Opacity reset (3000 iter마다): 불필요한 Gaussian 제거

In [ ]:
# === 셀 5: 학습 루프 (스냅샷 수집) ===

# 기록용 리스트
history = {
    "iterations": [0],
    "n_gaussians": [gaussians.get_xyz.shape[0]],
    "loss": [],
    "train_psnr": [],
    "test_psnr": [],
    "train_renders": [],  # (iter, image_numpy)
    "test_renders": [],
}

# 초기 스냅샷 (iter 0)
with torch.no_grad():
    tr = render(fixed_train_cam, gaussians, pipe, background)["render"]
    te = render(fixed_test_cam, gaussians, pipe, background)["render"]
    history["train_renders"].append((0, tr.permute(1, 2, 0).clamp(0, 1).cpu().numpy()))
    history["test_renders"].append((0, te.permute(1, 2, 0).clamp(0, 1).cpu().numpy()))

# 학습 루프
viewpoint_stack = train_cameras.copy()
ema_loss = 0.0

depth_l1_weight = get_expon_lr_func(opt.depth_l1_weight_init, opt.depth_l1_weight_final, max_steps=opt.iterations)

progress = tqdm(range(1, TOTAL_ITERATIONS + 1), desc="Training")
for iteration in progress:

    gaussians.update_learning_rate(iteration)

    # SH degree 증가 (1000마다)
    if iteration % 1000 == 0:
        gaussians.oneupSHdegree()

    # 랜덤 train 카메라 선택
    if not viewpoint_stack:
        viewpoint_stack = train_cameras.copy()
    idx = randint(0, len(viewpoint_stack) - 1)
    viewpoint_cam = viewpoint_stack.pop(idx)

    # 렌더링
    bg = background
    render_pkg = render(viewpoint_cam, gaussians, pipe, bg)
    image = render_pkg["render"]
    viewspace_point_tensor = render_pkg["viewspace_points"]
    visibility_filter = render_pkg["visibility_filter"]
    radii = render_pkg["radii"]

    # Loss 계산
    gt_image = viewpoint_cam.original_image.cuda()
    if viewpoint_cam.alpha_mask is not None:
        image *= viewpoint_cam.alpha_mask.cuda()

    Ll1 = l1_loss(image, gt_image)
    if FUSED_SSIM:
        ssim_val = fused_ssim(image.unsqueeze(0), gt_image.unsqueeze(0))
    else:
        ssim_val = ssim(image, gt_image)

    loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim_val)
    loss.backward()

    ema_loss = 0.4 * loss.item() + 0.6 * ema_loss

    with torch.no_grad():
        # Densification
        if iteration < opt.densify_until_iter:
            gaussians.max_radii2D[visibility_filter] = torch.max(
                gaussians.max_radii2D[visibility_filter], radii[visibility_filter])
            gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter)

            if iteration > opt.densify_from_iter and iteration % opt.densification_interval == 0:
                size_threshold = 20 if iteration > opt.opacity_reset_interval else None
                gaussians.densify_and_prune(opt.densify_grad_threshold, 0.005,
                                            scene.cameras_extent, size_threshold, radii)

            if iteration % opt.opacity_reset_interval == 0:
                gaussians.reset_opacity()

        # Optimizer step
        gaussians.exposure_optimizer.step()
        gaussians.exposure_optimizer.zero_grad(set_to_none=True)
        gaussians.optimizer.step()
        gaussians.optimizer.zero_grad(set_to_none=True)

    # 스냅샷 (SNAPSHOT_INTERVAL마다)
    if iteration % SNAPSHOT_INTERVAL == 0:
        with torch.no_grad():
            n_gauss = gaussians.get_xyz.shape[0]

            # 고정 시점 렌더링
            tr_img = render(fixed_train_cam, gaussians, pipe, background)["render"]
            te_img = render(fixed_test_cam, gaussians, pipe, background)["render"]

            # PSNR 계산
            tr_psnr = psnr(tr_img, fixed_train_cam.original_image.cuda()).item()
            te_psnr = psnr(te_img, fixed_test_cam.original_image.cuda()).item()

            history["iterations"].append(iteration)
            history["n_gaussians"].append(n_gauss)
            history["loss"].append(ema_loss)
            history["train_psnr"].append(tr_psnr)
            history["test_psnr"].append(te_psnr)
            history["train_renders"].append((iteration, tr_img.permute(1, 2, 0).clamp(0, 1).cpu().numpy()))
            history["test_renders"].append((iteration, te_img.permute(1, 2, 0).clamp(0, 1).cpu().numpy()))

        progress.set_postfix({
            "Loss": f"{ema_loss:.4f}",
            "Gaussians": f"{n_gauss:,}",
            "PSNR(train)": f"{tr_psnr:.1f}",
            "PSNR(test)": f"{te_psnr:.1f}",
        })

print(f"\nTraining complete. Final: {gaussians.get_xyz.shape[0]:,} gaussians")

In [ ]:
# === 셀 6: Gaussian 개수 변화 & Loss/PSNR 그래프 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Gaussian 개수 변화
ax = axes[0]
ax.plot(history["iterations"], history["n_gaussians"], "o-", color="purple", lw=2)
ax.axhline(history["n_gaussians"][0], color="gray", linestyle="--", alpha=0.5, label=f"Initial: {history['n_gaussians'][0]:,}")
ax.set_xlabel("Iteration")
ax.set_ylabel("Number of Gaussians")
ax.set_title("Densification: Gaussian Count")
ax.legend()
ax.grid(True, alpha=0.3)
# 주요 구간 표시
ax.axvspan(opt.densify_from_iter, opt.densify_until_iter, alpha=0.1, color="green", label="Densification zone")

# 2) Loss
ax = axes[1]
iters_loss = history["iterations"][1:]  # iter 0 에는 loss 없음
ax.plot(iters_loss, history["loss"], "o-", color="red", lw=2)
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss (EMA)")
ax.set_title("Training Loss")
ax.grid(True, alpha=0.3)

# 3) PSNR
ax = axes[2]
ax.plot(iters_loss, history["train_psnr"], "o-", color="steelblue", lw=2, label="Train PSNR")
ax.plot(iters_loss, history["test_psnr"], "s--", color="coral", lw=2, label="Test PSNR")
ax.set_xlabel("Iteration")
ax.set_ylabel("PSNR (dB)")
ax.set_title("Render Quality (PSNR)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f"Training Progress - {SCENE_NAME}", fontsize=14)
plt.tight_layout()
plt.show()

# 수치 테이블
print(f"\n{'Iter':>6} {'Gaussians':>12} {'Loss':>10} {'Train PSNR':>12} {'Test PSNR':>12}")
print("-" * 56)
print(f"{'0':>6} {history['n_gaussians'][0]:>12,} {'–':>10} {'–':>12} {'–':>12}")
for i, it in enumerate(iters_loss):
    print(f"{it:>6} {history['n_gaussians'][i+1]:>12,} {history['loss'][i]:>10.4f} {history['train_psnr'][i]:>12.2f} {history['test_psnr'][i]:>12.2f}")

In [ ]:
# === 셀 7: 학습 과정 렌더링 비교 (Train View vs Test View) ===
# 500 iteration마다 저장한 스냅샷을 시간순으로 나열

n_snapshots = len(history["train_renders"])
fig, axes = plt.subplots(3, n_snapshots, figsize=(4 * n_snapshots, 12))

# GT (첫 행)
train_gt_np = fixed_train_cam.original_image.permute(1, 2, 0).clamp(0, 1).cpu().numpy()
test_gt_np = fixed_test_cam.original_image.permute(1, 2, 0).clamp(0, 1).cpu().numpy()

for j in range(n_snapshots):
    it, tr_img = history["train_renders"][j]
    _, te_img = history["test_renders"][j]

    # Row 0: Train view render
    axes[0, j].imshow(tr_img)
    axes[0, j].set_title(f"iter {it}", fontsize=10)
    axes[0, j].axis("off")

    # Row 1: Test view render
    axes[1, j].imshow(te_img)
    axes[1, j].axis("off")

    # Row 2: GT (동일)
    if j == 0:
        axes[2, j].imshow(train_gt_np)
        axes[2, j].axis("off")
    elif j == n_snapshots - 1:
        axes[2, j].imshow(test_gt_np)
        axes[2, j].axis("off")
    else:
        axes[2, j].axis("off")

# Row labels
axes[0, 0].set_ylabel("Train View\n(Render)", fontsize=12, rotation=0, labelpad=80, va="center")
axes[1, 0].set_ylabel("Test View\n(Render)", fontsize=12, rotation=0, labelpad=80, va="center")
axes[2, 0].set_ylabel("GT", fontsize=12, rotation=0, labelpad=80, va="center")

# GT 표시
axes[2, 0].set_title("Train GT", fontsize=10)
axes[2, n_snapshots-1].set_title("Test GT", fontsize=10)

plt.suptitle(f"Rendering Progress - {SCENE_NAME} ({SNAPSHOT_INTERVAL} iter intervals)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === 셀 8: 학습 후 Gaussian 속성 변화 확인 ===
# 학습 전(셀 3) vs 학습 후 비교

n_final = gaussians.get_xyz.shape[0]

with torch.no_grad():
    opacity_final = gaussians.get_opacity.cpu().numpy()
    scales_final = gaussians.get_scaling.cpu().numpy().mean(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Opacity 분포 변화
ax = axes[0]
ax.hist(opacity_final, bins=50, color="coral", edgecolor="white", alpha=0.8)
ax.axvline(opacity_final.mean(), color="red", linestyle="--", label=f"Mean: {opacity_final.mean():.3f}")
ax.set_xlabel("Opacity")
ax.set_title(f"Opacity After Training\n({n_final:,} gaussians)")
ax.legend()

# Scale 분포
ax = axes[1]
ax.hist(scales_final, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ax.set_xlabel("Scale (mean)")
ax.set_title("Scale Distribution After Training")

# 3D 포인트 비교 (before vs after)
ax = axes[2]
xyz_final = gaussians.get_xyz.detach().cpu().numpy()
sub_f = max(1, len(xyz_final) // 3000)
ax.scatter(xyz_final[::sub_f, 0], xyz_final[::sub_f, 2], s=0.3, alpha=0.3, c="coral", label=f"After ({n_final:,})")
xyz_init = history["train_renders"][0]  # 참고용
ax.set_title(f"Point Cloud: {history['n_gaussians'][0]:,} → {n_final:,}")
ax.set_xlabel("X"); ax.set_ylabel("Z")
ax.set_aspect("equal")
ax.legend()

plt.suptitle(f"Gaussian Properties After {TOTAL_ITERATIONS} Iterations", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\n=== Summary ===")
print(f"  Initial Gaussians: {history['n_gaussians'][0]:,}")
print(f"  Final Gaussians:   {n_final:,} ({n_final/history['n_gaussians'][0]:.1f}x)")
print(f"  Mean Opacity:      {opacity_final.mean():.4f}")
print(f"  Low Opacity (<0.1): {(opacity_final < 0.1).sum()} ({100*(opacity_final < 0.1).mean():.1f}%)")